# Avance 3. Modelos de referencia (*baseline*)

**Inferencia de propiedades físicas de galaxias a partir de imágenes multibanda**

Instituto Tecnológico y de Estudios Superiores de Monterrey · Maestría en Inteligencia Artificial Aplicada · **Equipo 24**

Colaboración científica con el Instituto de Astronomía de la UNAM

---

| Integrante | Matrícula |
|---|---|
| José Florencio Maguey Peralta | A01796727 |
| Erick Alberto Bustos Cruz | A01378966 |
| Luz Copelia Minutti Pérez | A01796921 |

**Asesores:** Dr. Iván Reyes Amezcua (ITESM) · Dr. Héctor Hernández Toledo y Dr. José Antonio Vázquez Mata (IA-UNAM)

---

**Fase de CRISP-ML(Q):** 3, *Machine Learning Model Engineering* (Studer et al. 2021).

**Objetivos del avance:**

- **3.1** Establecer las medidas de calidad del modelo de aprendizaje automático.
- **3.2** Proporcionar un marco de referencia para evaluar y mejorar modelos más avanzados.

## 1. Planteamiento

El proyecto estima tres propiedades de cada galaxia a partir de su imagen del DESI Legacy Imaging Survey en las bandas g, r y z (Dey et al. 2019):

| Propiedad | Qué mide | Unidad |
|---|---|---|
| **Masa estelar**, log M\* | Cuánta masa hay en estrellas | log₁₀ de masas solares |
| **Tasa de formación estelar**, log SFR | Cuántas estrellas nuevas forma la galaxia por año | log₁₀ de masas solares por año |
| **D4000** | Intensidad del salto espectral en 4000 Å; crece con la edad de las estrellas | adimensional (≈ 1 en poblaciones jóvenes, ≈ 2 en viejas) |

Las etiquetas provienen de Pipe3D (Sánchez et al. 2022), que mide estas propiedades con la espectroscopía de campo integral de MaNGA. Esa medición es costosa y existe para unas diez mil galaxias, mientras que las imágenes existen para cientos de millones. Un modelo que recupere las propiedades desde la imagen permitiría extender estimaciones de calidad espectroscópica a todo el cielo observado.

Antes de invertir en modelos complejos, este avance responde dos preguntas:

> **¿Es viable estimar estas propiedades con métodos simples, y cuál es el desempeño mínimo que un modelo más avanzado tendrá que superar?**

### 1.1 Los modelos de referencia

Construimos una escalera de referencias, de menor a mayor complejidad. Cada peldaño responde una pregunta distinta:

| Peldaño | Modelo | Entrada | Pregunta que responde |
|---|---|---|---|
| **Piso** | Predecir siempre la media de entrenamiento | Ninguna | ¿Cuánto error comete un modelo sin información? Es el equivalente del azar |
| **B1. Relación color–masa** | Regresión de una variable con la forma de Taylor et al. (2011) | Color g−i y magnitud absoluta Mᵢ de la fotometría integrada | ¿Cuánto se logra con el método clásico de la astronomía, sin imágenes y casi sin ajuste? |
| **B2. Red convolucional** | Cadena de tres ResNet50 de Katachi (Alfonzo et al. 2024), entrenada con nuestras imágenes DESI | La imagen de 300 × 300 píxeles | ¿Cuánto se logra con un método publicado que usa la imagen completa? Es **el baseline del proyecto** |
| Referencia histórica | Predicciones publicadas de Katachi, galaxia por galaxia | Imágenes de SDSS | ¿Cómo se compara nuestro baseline con el estado actual del problema? |

B1 y B2 son modelos de referencia de naturaleza distinta y se complementan. B1 indica qué parte de la masa ya está contenida en dos números por galaxia; B2 indica qué tan lejos llega un modelo que solo ve la imagen. Los modelos que se desarrollen después deberán superar a B2.

### 1.2 Criterios generales

- **Muestra única.** Todas las comparaciones usan las mismas galaxias y la misma partición: entrenamiento, validación y prueba (QCFLAG = 0, partición `split_proyecto`). Cuando un modelo no cubre todas las galaxias, la comparación se hace sobre las que comparten.
- **Decisiones con validación.** La elección de variantes, los criterios de éxito y los umbrales se fijan con entrenamiento y validación, y se registran antes de abrir el conjunto de prueba.
- **Prueba una sola vez.** El conjunto de prueba se abre en la sección 10, después del registro.
- **Incertidumbre.** Cada métrica se reporta con su intervalo de confianza del 95 % por *bootstrap* de galaxias, y la red se entrena con tres semillas.

## 2. Parámetros

| Parámetro | Valor | Significado |
|---|---|---|
| `SEMILLAS` | 42, 7, 2026 | Semillas de la red y de las particiones de validación cruzada. La primera es la principal, fijada antes de ver resultados |
| `EPOCAS` | 30 | Épocas de entrenamiento de la red |
| `N_BOOT` | 2000 | Remuestreos *bootstrap* para los intervalos de confianza |
| `CORTE_SSFR` | −10.8 | Frontera entre galaxias formadoras y apagadas, en log sSFR = log SFR − log M\* [año⁻¹] |
| `MAG_MIN`, `MAG_MAX` | −25, −14 | Intervalo válido de las magnitudes absolutas g, r, i |
| `INCL_MAX` | 70° | Inclinación máxima, con cos i = b/a |
| `TAYLOR` | 1.15, 0.70 | Coeficientes publicados de la relación color–masa (Taylor et al. 2011) |
| `UMBRAL_CATASTROFICO` | 0.3 dex | Un error mayor equivale a errar por más de un factor 2 |
| `MARGEN` | 0.02, 0.02, 0.01 | Diferencia de RMSE que se considera despreciable en log M\*, log SFR y D4000 |
| `N_SHAP` | 48 | Galaxias de validación para los mapas SHAP |

In [ ]:
import os
env = os.environ.get            # las variables AV3_* solo se usan en la prueba local reducida

SEMILLAS = tuple(int(s) for s in env("AV3_SEMILLAS", "42,7,2026").split(","))
SEMILLA = SEMILLAS[0]
EPOCAS = int(env("AV3_EPOCAS", "30"))
N_BOOT = int(env("AV3_N_BOOT", "2000"))
LIMITE = int(env("AV3_LIMIT", "0"))             # 0 = todas las galaxias
N_SHAP = int(env("AV3_N_SHAP", "48"))
SHAP_MUESTRAS = int(env("AV3_SHAP_NSAMPLES", "200"))

CORTE_SSFR = -10.8
MAG_MIN, MAG_MAX = -25.0, -14.0
INCL_MAX = 70.0
H_NSA = 0.7                                     # las masas de NSA están en unidades de h⁻² M☉
TAYLOR = (1.15, 0.70)
UMBRAL_CATASTROFICO = 0.3
OBJETIVOS = ["log_mstar", "log_sfr", "d4000"]
NOMBRES = {"log_mstar": "log M* [dex]", "log_sfr": "log SFR [dex]", "d4000": "D4000"}
CORTO = {"log_mstar": "M*", "log_sfr": "SFR", "d4000": "D4000"}
MARGEN = {"log_mstar": 0.02, "log_sfr": 0.02, "d4000": 0.01}
KATACHI_PUBLICADO = {"log_mstar": 0.222, "log_sfr": 0.668, "d4000": 0.100}   # RMSE en sus galaxias de prueba
MASA_DESEABLE = 0.15                            # precisión típica de las masas estelares (dex)

## 3. Entorno

Descargamos el código del repositorio del equipo, de modo que cada resultado quede ligado a una versión concreta, y montamos Google Drive, donde están el catálogo, las imágenes y los resultados. La red necesita GPU.

In [ ]:
import sys, json, time, hashlib, subprocess, platform, warnings
from pathlib import Path
from dataclasses import replace
from datetime import datetime, timezone

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/erteck/InferenciaPropiedadesFisicas.git"
REPO_DIR = Path(env("AV3_REPO_DIR", "/content/InferenciaPropiedadesFisicas"))
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hickle", "shap"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
sys.path.insert(0, str(REPO_DIR))
COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import ndimage
from IPython.display import display, Markdown
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

from src.katachi_fits import config as K, catalog as C, diagnostico as DG, shap_maps as SM
from src.katachi_fits import aumentacion as AU
from src.katachi_fits.cache import build_cache, cache_dir, load_cache
from src.katachi_fits.chain import ChainTarget
from src.katachi_fits.config import FitsConfig, MejorasConfig, Paths, TrainConfig
from src.katachi_fits.experimentos import Experimento, correr, preparar_datos
from src.katachi_fits.io_guard import WriteGuard, sha256
from src.katachi_fits.train import GPUData, Trainer
from src.katachi_fits.transforms import Normalizer, band_permutation
from src.katachi_fits.variants import DESI

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.precision", 3, "display.max_columns", 40, "display.width", 220)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
COLOR = {"Piso": "0.6", "B1": "#2a9d8f", "B2": "#264653", "Katachi": "#e76f51"}
print("Python", platform.python_version(), "· commit", COMMIT[:12], "·", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

## 4. Datos

### 4.1 Muestra y partición

La muestra son las galaxias de MaNGA con etiquetas de Pipe3D que pasaron todos sus controles de calidad (QCFLAG = 0) y cumplen los cortes de Katachi: 8 < log M\* < 12, −4 < log SFR < 2 y D4000 definido. El conjunto de prueba coincide con el de Katachi, lo que permite comparar con sus predicciones publicadas galaxia por galaxia.

In [ ]:
BASE = Path(env("AV3_BASE", str(K.DRIVE_BASE)))
DESCARGA = BASE / K.DOWNLOAD_DIRNAME
PLAN_CSV = next(DESCARGA.glob("**/plan.csv"))
CATALOGO = next(p for n in K.TEAM_CATALOG_NAMES for p in DESCARGA.glob(f"**/{n}"))
paths = Paths(base=BASE, plan_csv=PLAN_CSV, inventario_csv=PLAN_CSV.parent / "inventario.csv",
              fits_dir=DESCARGA / "fits", results=BASE / K.RESULTS_DIRNAME, team_catalog=CATALOGO)
guard = WriteGuard(paths.results, paths.read_only)
print("Catálogo:", CATALOGO.name, "· ¿versión acordada?", sha256(CATALOGO) == K.TEAM_CATALOG_SHA256)

plan = C.load_plan(PLAN_CSV, paths.fits_dir)
equipo = C.load_team_catalog(CATALOGO)
scalars = C.load_scalars(paths.external / "scalars.cat")
cohorte, _ = C.build_cohort_team(plan, equipo, scalars)
if LIMITE:
    cohorte = (cohorte.groupby("particion", group_keys=False)
                      .apply(lambda d: d.head(max(LIMITE // 3, 12) if d.name != "Train" else LIMITE)).reset_index(drop=True))
cohorte, _ = C.apply_qc(cohorte, K.QC_GROUPS_INCLUDED)
cohorte = cohorte.reset_index(drop=True)
TAG = f"_lim{LIMITE}" if LIMITE else ""
RES = guard.mkdir(paths.results / f"avance3{TAG}")

cohorte["log_ssfr"] = cohorte.log_sfr - cohorte.log_mstar
cohorte["poblacion"] = np.where(cohorte.log_ssfr < CORTE_SSFR, "apagada", "formadora")
IDX = {p: np.where(cohorte.particion == p)[0] for p in ("Train", "Validation", "Test")}
Y = cohorte[OBJETIVOS].to_numpy(float)
display(pd.crosstab(cohorte.particion, cohorte.poblacion, margins=True, margins_name="total")
          .loc[["Train", "Validation", "Test", "total"]])

### 4.2 Las imágenes

Cada galaxia es un recorte central de 300 × 300 píxeles (78.6″) de los coadds de DESI DR10 en g, r y z. El flujo se comprime con asinh(x / 3σ) y se divide entre una constante por banda calculada en entrenamiento. Son las mismas constantes para todas las galaxias, de modo que la red conserva el brillo absoluto de cada una, necesario para estimar la masa.

También medimos el **radio que encierra la mitad de la luz** (R50) en la banda r. Las secciones de importancia expresan las regiones de la imagen en unidades de R50, porque las galaxias tienen tamaños muy distintos.

In [ ]:
fcfg = FitsConfig()
CACHE = cache_dir(paths.cache, DESI, fcfg, TAG)
build_cache(cohorte, "ruta_local", CACHE, guard, fcfg, variant=DESI, chunk=500, workers=24)
X300, ids, _, _ = load_cache(CACHE, dtype=fcfg.storage_dtype)
assert list(ids) == list(cohorte.mangaid), "El orden de la caché no coincide con la muestra"
NORM = Normalizer.from_json((paths.results / "normalizacion" / f"normalizador_recorte300_beta3{TAG}.json").read_text())
PERM = band_permutation(K.INPUT_BAND_ORDER)
BANDAS_RED = list(K.INPUT_BAND_ORDER)                     # orden de los canales que recibe la red
R50 = DG.radio_mitad_luz(X300, banda=list(K.BANDS).index("r"))
cohorte["R50_px"] = R50
print("Imágenes:", X300.shape, "· σ del cielo:", np.round(NORM.sigma, 5), "· escala:", np.round(NORM.scale, 3))
print(f"R50 en la banda r: mediana {np.median(R50):.1f} px ({np.median(R50) * K.DESI_PIXSCALE_ARCSEC:.1f}″)")

def vista(i):
    x = AU.estirar(torch.from_numpy(X300[i:i + 1].astype(np.float32)), torch.as_tensor(NORM.sigma, dtype=torch.float32),
                   NORM.beta, torch.as_tensor(NORM.scale, dtype=torch.float32))[:, PERM]
    return DG.vista_rgb(x[0].numpy())

tr = cohorte.iloc[IDX["Train"]]
ej = [tr.index[np.argmin(np.abs(tr.log_mstar.to_numpy() - q))] for q in np.quantile(tr.log_mstar, np.linspace(0.03, 0.97, 8))]
fig, ax = plt.subplots(1, 8, figsize=(16, 2.6))
for a, i in zip(ax, ej):
    f = cohorte.loc[i]
    a.imshow(vista(i), origin="lower")
    a.add_patch(plt.Circle((149.5, 149.5), f.R50_px, fill=False, color="w", lw=0.8, ls="--"))
    a.set_title(f"log M* {f.log_mstar:.1f} · SFR {f.log_sfr:.1f}\nD4000 {f.d4000:.2f} · z {f.redshift:.3f}", fontsize=7)
    a.set_xticks([]); a.set_yticks([])
plt.suptitle("Galaxias de entrenamiento ordenadas por masa estelar (canales z, r, g como rojo, verde y azul; círculo: R50)", y=1.04)
plt.show()

### 4.3 Fotometría integrada

El modelo B1 usa la fotometría integrada del NASA–Sloan Atlas (NSA; Blanton et al. 2011), basada en SDSS: las magnitudes absolutas corregidas en g, r e i, el cociente de ejes b/a y la masa estelar que el propio NSA calcula con esa fotometría.

Las relaciones color–masa se calibran con galaxias de fotometría confiable, por lo que solo se aplican dentro de su dominio de validez:

- asociación con el NSA y magnitudes g, r, i finitas;
- magnitudes absolutas entre −25 y −14;
- inclinación de hasta 70°. En los discos muy inclinados el polvo enrojece el color y la relación sobreestima la masa.

La fracción de galaxias que cumple estos requisitos es la **cobertura** de B1. Es una característica del método: la red estima todas las galaxias que tienen imagen.

In [ ]:
NOMBRE_NSA = "tabla_maestra_sdss_nsa.csv"
def _buscar_nsa():
    for p in (BASE / "entregable3_insumos" / NOMBRE_NSA, BASE / NOMBRE_NSA):
        if p.is_file():
            return p
    for patron in (f"*/{NOMBRE_NSA}", f"*/*/{NOMBRE_NSA}"):
        p = next((q for q in BASE.glob(patron) if q.is_file()), None)
        if p is not None:
            return p
    return None
RUTA_NSA = _buscar_nsa()
print("Fotometría integrada:", RUTA_NSA)
assert RUTA_NSA is not None, f"No se encontró {NOMBRE_NSA} en Drive (se espera en MyDrive/entregable3_insumos/)"
nsa = pd.read_csv(RUTA_NSA, dtype={"MANGAID": "string"})
nsa = nsa.assign(mangaid=nsa.MANGAID.str.strip()).drop_duplicates("mangaid")
num = lambda c: pd.to_numeric(nsa[c], errors="coerce")
nsa = pd.DataFrame({"mangaid": nsa.mangaid, "M_g": num("Mabs_g_corr"), "M_r": num("Mabs_r_corr"), "M_i": num("Mabs_i_corr"),
                    "b_a": num("NSA_SERSIC_BA"), "masa_nsa": num("NSA_SERSIC_MASS")})
cohorte = cohorte.drop(columns=[c for c in nsa.columns if c != "mangaid"], errors="ignore").merge(nsa, on="mangaid", how="left")
cohorte["g_i"] = cohorte.M_g - cohorte.M_i
cohorte["logM_nsa"] = np.log10(cohorte.masa_nsa.where(cohorte.masa_nsa > 0)) - 2 * np.log10(H_NSA)
cohorte["inclinacion"] = np.degrees(np.arccos(cohorte.b_a.clip(0, 1)))

mags = cohorte[["M_g", "M_r", "M_i"]]
pasos = [("Con imagen y etiquetas", np.ones(len(cohorte), bool)),
         ("Asociada al NSA, con masa NSA", np.isfinite(cohorte.logM_nsa)),
         ("Magnitudes g, r, i finitas", np.isfinite(mags).all(1)),
         (f"Magnitudes entre {MAG_MIN:g} y {MAG_MAX:g}", mags.ge(MAG_MIN).all(1) & mags.le(MAG_MAX).all(1)),
         (f"Inclinación ≤ {INCL_MAX:g}°", cohorte.inclinacion.le(INCL_MAX))]
vigente = np.ones(len(cohorte), bool)
filas = []
for nombre, cond in pasos:
    vigente = vigente & np.asarray(cond, bool)
    filas.append({"requisito": nombre, **{p: int(vigente[IDX[p]].sum()) for p in IDX}})
cohorte["cubierta"] = vigente
COB = pd.DataFrame(filas).set_index("requisito")
COB.loc["Cobertura de B1"] = [f"{100 * cohorte.cubierta.iloc[IDX[p]].mean():.0f} %" for p in IDX]
display(COB)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for a, t in zip(ax, OBJETIVOS):
    b = np.linspace(*np.nanpercentile(cohorte[t], [0.5, 99.5]), 35)
    a.hist(cohorte.loc[cohorte.cubierta, t], b, density=True, alpha=0.6, color=COLOR["B1"], label="cubiertas por B1")
    a.hist(cohorte.loc[~cohorte.cubierta, t], b, density=True, histtype="step", lw=1.5, color="k", label="fuera del dominio")
    a.set(xlabel=NOMBRES[t], ylabel="densidad")
ax[0].legend(fontsize=7); plt.suptitle("¿Las galaxias fuera del dominio de B1 son distintas?", y=1.02); plt.tight_layout(); plt.show()

**Por qué lo hacemos.** B1 depende de una fotometría que no existe para todas las galaxias, y conviene saber cuántas cubre y si las que quedan fuera son de un tipo particular. Si las excluidas fueran, por ejemplo, las menos masivas, el error de B1 parecería menor de lo que sería en la muestra completa.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 4.4 Las etiquetas

El plano SFR–M\* resume la física que el modelo debe aprender. Las galaxias formadoras siguen una secuencia en la que la SFR crece con la masa; las apagadas, más viejas (D4000 alto), forman muy pocas estrellas y quedan por debajo. Esa bimodalidad condiciona la elección de la métrica (sección 5).

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 3.6), gridspec_kw={"width_ratios": [1.3, 1, 1, 1]})
hb = ax[0].hexbin(cohorte.log_mstar, cohorte.log_sfr, C=cohorte.d4000, reduce_C_function=np.median, gridsize=45,
                  cmap="RdYlBu_r", mincnt=2)
m = np.linspace(8, 12, 10); ax[0].plot(m, m + CORTE_SSFR, "k--", lw=1, label="log sSFR = −10.8")
ax[0].set(xlabel=NOMBRES["log_mstar"], ylabel=NOMBRES["log_sfr"], title="Plano SFR–M* (color: D4000 mediano)")
plt.colorbar(hb, ax=ax[0], label="D4000"); ax[0].legend(fontsize=7, loc="lower right")
for a, t in zip(ax[1:], OBJETIVOS):
    b = np.linspace(*np.percentile(cohorte[t], [0.5, 99.5]), 40)
    for pob, c in (("formadora", "#457b9d"), ("apagada", "#e63946")):
        a.hist(cohorte.loc[cohorte.poblacion == pob, t], b, alpha=0.6, color=c, label=pob)
    a.set(xlabel=NOMBRES[t], ylabel="galaxias")
ax[1].legend(fontsize=7); plt.tight_layout(); plt.show()

err = {"log_mstar": "error_log_mstar", "log_sfr": "error_log_sfr", "d4000": "error_d4000"}
ET = pd.DataFrame({NOMBRES[t]: {
    "σ de la etiqueta (entrenamiento)": cohorte[t].iloc[IDX["Train"]].std(),
    "error típico de Pipe3D (mediana)": cohorte[err[t]].median(),
    "error de Pipe3D, percentil 90": cohorte[err[t]].quantile(0.9)} for t in OBJETIVOS})
display(ET.round(3))
RUIDO = {t: float(cohorte[err[t]].median()) for t in OBJETIVOS}
SIGMA = {t: float(cohorte[t].iloc[IDX["Validation"]].std()) for t in OBJETIVOS}

**Por qué lo hacemos.** Las etiquetas no son exactas: Pipe3D reporta un error para cada una. Ese error es el límite de lo que se puede exigir a cualquier modelo, porque un modelo que predijera la propiedad verdadera sin error aún mostraría esa diferencia frente a la etiqueta.

**Qué observamos.** _Se completa con los resultados de la corrida._

## 5. Métrica

### 5.1 Métrica principal: RMSE en dex

Elegimos la **raíz del error cuadrático medio (RMSE)** por tres razones ligadas al uso del catálogo:

1. **Interpretación directa.** log M\* y log SFR están en logaritmo, así que un error de Δ dex equivale a errar por un factor 10^Δ. Un RMSE de 0.22 dex significa un error típico de un factor 1.7; uno de 0.12 dex, de un factor 1.3.
2. **Penaliza los errores grandes.** Al elevar al cuadrado, una galaxia apagada que se estima como formadora (un error de 1 a 2 dex en SFR) pesa mucho más que muchos errores pequeños. Esos errores grandes son los que mezclan poblaciones y deforman los estudios que se hagan con el catálogo.
3. **Comparabilidad.** Es la métrica de Katachi y de la literatura de estimación de propiedades de galaxias (Géron 2022, cap. 2, la recomienda como medida por defecto en regresión cuando los errores grandes son costosos).

### 5.2 Métricas complementarias

| Métrica | Definición | Para qué sirve en este problema |
|---|---|---|
| **Sesgo** | Media de (predicción − etiqueta) | Un error sistemático desplaza relaciones físicas completas, como la secuencia principal o la función de masa |
| **MAE** | Media de \|predicción − etiqueta\| | Error típico, menos sensible a los valores extremos que el RMSE |
| **σ_NMAD** | 1.4826 · mediana(\|r − mediana(r)\|) | Dispersión del núcleo de la distribución, estándar en las estimaciones fotométricas de astronomía (Brammer et al. 2008) |
| **Catastróficos** | Fracción con \|error\| > 0.3 dex (un factor 2) | Proporción de galaxias que el catálogo ubicaría en la población equivocada |
| **R²** | 1 − SSE/SST | Solo como referencia, porque depende de la variedad de la muestra |

### 5.3 Particularidades de los datos

En este problema no hay clases, pero hay un desbalance equivalente:

- **La SFR es bimodal.** Casi la mitad de las galaxias están apagadas y su SFR es muy baja y difícil de medir. Un RMSE global puede verse bien aunque el modelo confunda las dos poblaciones.
- **Los extremos están poco representados.** Hay pocas galaxias de masa baja y pocas muy cercanas, y son justamente las que más se alejan de la media.

Por eso, además del valor global, reportamos las métricas **por población** (formadoras y apagadas) y **por intervalo de masa**.

In [ ]:
BORDES_M = [8, 9.5, 10, 10.5, 11, 12]
cohorte["bin_masa"] = pd.cut(cohorte.log_mstar, BORDES_M)
display(pd.crosstab(cohorte.bin_masa, cohorte.poblacion).rename_axis("log M*").T)

def evaluar(y, p, t):
    return DG.metricas(y, p, UMBRAL_CATASTROFICO if t != "d4000" else None)

def tabla(modelos, idx, mascara=None, ic=True):
    # Métricas de varios modelos sobre las mismas galaxias. modelos: {nombre: predicciones (N, 3)}.
    sel = idx if mascara is None else idx[mascara[idx]]
    filas = []
    for nombre, P in modelos.items():
        for k, t in enumerate(OBJETIVOS):
            y, p = Y[sel, k], P[sel, k]
            ok = np.isfinite(p)
            m = evaluar(y[ok], p[ok], t)
            f = {"modelo": nombre, "propiedad": CORTO[t], **m}
            if ic:
                lo, hi = DG.rmse_ic(y[ok], p[ok], n=N_BOOT)
                f["IC 95 %"] = f"[{lo:.3f}, {hi:.3f}]"
            filas.append(f)
    return pd.DataFrame(filas)

def ordenar(df, niveles):
    # Orden de lectura: propiedad (M*, SFR, D4000) y modelo (Piso, B1, Katachi, B2, semillas).
    orden = {"galaxias": ["todas", "cubiertas por B1"], "propiedad": ["M*", "SFR", "D4000"], "modelo": ["Piso", "B1", "Katachi", "B2"] + [f"B2 semilla {s}" for s in SEMILLAS]}
    d = df.copy()
    for c in niveles:
        if c in orden:
            d[c] = pd.Categorical(d[c], [v for v in orden[c] if v in set(d[c])], ordered=True)
    return d.sort_values(niveles).set_index(niveles)

def veredicto(lo, hi, margen):
    if hi < 0:
        return "mejor"
    if lo > 0:
        return "peor"
    if -margen < lo and hi < margen:
        return "equivalente"
    return "no concluyente"

**Por qué lo hacemos.** La tabla muestra cuántas galaxias hay en cada intervalo de masa y población. Los grupos pequeños son donde un promedio global puede ocultar un mal desempeño.

**Qué observamos.** _Se completa con los resultados de la corrida._

## 6. Desempeño mínimo

El problema tiene antecedentes, así que el desempeño mínimo no se fija de forma arbitraria. Reunimos cuatro referencias:

| Referencia | Qué representa |
|---|---|
| **σ de la etiqueta** | El error del piso: predecir siempre la media. Un modelo útil debe estar claramente por debajo |
| **Katachi publicado** (Alfonzo et al. 2024) | El desempeño histórico al estimar estas mismas etiquetas desde imágenes, con SDSS |
| **Precisión típica de las masas estelares**: 0.10–0.15 dex | La dispersión entre métodos establecidos de estimación de masas a partir de colores y del ajuste de distribuciones espectrales (Roediger y Courteau 2015) |
| **Error de las etiquetas** (Pipe3D) | El límite inferior alcanzable |

Con ellas fijamos los criterios **antes** de evaluar los modelos y los guardamos en `criterios.json`:

| Criterio | Condición | Sobre |
|---|---|---|
| **Viabilidad** | El intervalo del 95 % del RMSE queda completamente por debajo del RMSE del piso en las mismas galaxias | B1 y B2, en cada propiedad |
| **Desempeño mínimo del baseline** | B2 no es peor que Katachi: el límite superior del intervalo de ΔRMSE = RMSE(B2) − RMSE(Katachi) es menor que el margen | B2 frente a Katachi en las mismas galaxias de prueba, con las tres semillas |
| **Desempeño deseable en masa** | RMSE ≤ 0.15 dex | B2, masa estelar |

In [ ]:
REF = pd.DataFrame({CORTO[t]: {"σ de la etiqueta (validación)": SIGMA[t], "Katachi publicado (prueba)": KATACHI_PUBLICADO[t],
                               "Precisión típica de las masas": MASA_DESEABLE if t == "log_mstar" else np.nan,
                               "Error de la etiqueta (Pipe3D)": RUIDO[t]} for t in OBJETIVOS})
display(REF.round(3))
CRITERIOS = {"viabilidad": "IC95 superior del RMSE < RMSE del piso en las mismas galaxias", "minimo": "IC95 superior de ΔRMSE(B2 − Katachi) < margen",
             "deseable_masa": MASA_DESEABLE, "margen": MARGEN, "sigma": SIGMA, "katachi_publicado": KATACHI_PUBLICADO,
             "semillas": SEMILLAS, "fecha_utc": datetime.now(timezone.utc).isoformat(), "commit": COMMIT}
f_crit = RES / "criterios.json"
if f_crit.exists():
    anterior = json.loads(f_crit.read_text())
    assert all(anterior[k] == json.loads(json.dumps(CRITERIOS[k])) for k in ("viabilidad", "minimo", "deseable_masa", "margen")), \
        "Los criterios cambiaron después de registrarse"
    print("Criterios registrados el", anterior["fecha_utc"])
else:
    guard.write_json(f_crit, CRITERIOS)
    print("Criterios registrados en", f_crit)

## 7. Algoritmos y resultados en validación

### 7.1 Por qué estos algoritmos

El tipo de dato define el algoritmo. B1 trabaja con **datos estructurados** de dos variables con una relación física conocida; B2, con **datos no estructurados**: imágenes de tres bandas.

| Criterio | Regresión lineal (B1) | Bosque aleatorio sobre las mismas variables | Red desde cero | **ResNet50 con transferencia (B2)** | Transformadores de visión o modelos fundacionales |
|---|---|---|---|---|---|
| Tipo de datos | Tabular | Tabular | Imagen | **Imagen** | Imagen |
| Cantidad de datos (7,123 galaxias) | Suficiente: 2 parámetros | Suficiente | Insuficiente para aprender filtros desde cero | **Suficiente: reutiliza filtros aprendidos en ImageNet** (Géron 2022, cap. 14) | Requieren más datos o un preentrenamiento astronómico; son candidatos para la siguiente fase |
| Relaciones no lineales | No | Sí | Sí | **Sí** | Sí |
| Estructura espacial (bulbo, disco, brazos) | No la ve | No la ve | Sí | **Sí, con invariancia a la traslación** | Sí |
| Interpretabilidad | **Máxima: coeficientes con significado físico** | Importancias | Baja | Media: oclusión y SHAP | Baja |
| Antecedente publicado | Taylor et al. (2011) | — | — | **Katachi** (Alfonzo et al. 2024) | Parcial |
| Costo | Segundos | Segundos | Horas | **~20 min por semilla en una A100** | Mayor |

**B1** usa la forma de Taylor et al. (2011). La masa es el producto de la luminosidad por el cociente masa–luminosidad (M/L), y el M/L depende del color:

log M\* = log(M/L) + log L = A + B (g − i) − 0.4 Mᵢ

El término −0.4 Mᵢ es la definición de la luminosidad en magnitudes, así que **su coeficiente está fijo**: solo se ajustan A y B, en el espacio log M\* + 0.4 Mᵢ frente a g − i. Para la SFR y el D4000 no existe una relación equivalente, y usamos una regresión lineal con las mismas dos variables. Como control de no linealidad ajustamos también un bosque aleatorio (Breiman 2001) con las mismas entradas y el mismo término fijo. Elegimos el modelo más simple cuyo error de validación cruzada no supera al mínimo en más de un error estándar (Hastie et al. 2009, §7.10).

**B2** es la cadena de Katachi: tres ResNet50 (He et al. 2016) preentrenadas en ImageNet que estiman, en orden, la masa, la SFR y el D4000. Cada red recibe la imagen y, como planos constantes adicionales, las predicciones de las anteriores, porque la SFR y la edad de una galaxia dependen de su masa.

### 7.2 B1: relación color–masa

Primero comparamos los coeficientes de Taylor con los que resultan al ajustar la relación con dos masas distintas: la del NSA, calculada con la misma fotometría, y la de Pipe3D, medida con el espectro, que es la etiqueta del proyecto.

In [ ]:
cub = cohorte.cubierta.to_numpy()
TRc, VAc = IDX["Train"][cub[IDX["Train"]]], IDX["Validation"][cub[IDX["Validation"]]]
GI, MI = cohorte.g_i.to_numpy(), cohorte.M_i.to_numpy()

def ajustar_b1(modelo, t, idx, seed=SEMILLA, objetivo=None):
    # Devuelve una función pred(gi, Mi). En masa, el término −0.4 Mi es fijo.
    y = (cohorte[t] if objetivo is None else objetivo).to_numpy()[idx]
    masa = t == "log_mstar"
    if modelo == "Taylor":
        return lambda gi, mi: TAYLOR[0] + TAYLOR[1] * gi - 0.4 * mi
    X = (GI[idx, None] if masa else np.c_[GI[idx], MI[idx]])
    yy = y + 0.4 * MI[idx] if masa else y
    est = (LinearRegression() if modelo == "Lineal" else
           RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=seed, n_jobs=-1)).fit(X, yy)
    if masa:
        f = lambda gi, mi: est.predict(np.asarray(gi)[:, None]) - 0.4 * np.asarray(mi)
    else:
        f = lambda gi, mi: est.predict(np.c_[gi, mi])
    f.est = est
    return f

coef = {}
for nombre, objetivo in (("NSA", cohorte.logM_nsa), ("Pipe3D", cohorte.log_mstar)):
    lr = ajustar_b1("Lineal", "log_mstar", TRc, objetivo=objetivo).est
    p = lr.predict(GI[VAc, None]) - 0.4 * MI[VAc]
    for etiqueta, yv in (("NSA", cohorte.logM_nsa.to_numpy()[VAc]), ("Pipe3D", Y[VAc, 0])):
        coef.setdefault(f"ajustada con masa {nombre}", {"A": lr.intercept_, "B": lr.coef_[0]})[f"RMSE frente a {etiqueta}"] = \
            DG.metricas(yv, p)["RMSE"]
pt = TAYLOR[0] + TAYLOR[1] * GI[VAc] - 0.4 * MI[VAc]
coef["Taylor et al. (2011), publicada"] = {"A": TAYLOR[0], "B": TAYLOR[1],
                                           "RMSE frente a NSA": DG.metricas(cohorte.logM_nsa.to_numpy()[VAc], pt)["RMSE"],
                                           "RMSE frente a Pipe3D": DG.metricas(Y[VAc, 0], pt)["RMSE"]}
COEF = pd.DataFrame(coef).T[["A", "B", "RMSE frente a NSA", "RMSE frente a Pipe3D"]]
display(COEF.round(3))

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
gx = np.linspace(np.nanpercentile(GI[TRc], 1), np.nanpercentile(GI[TRc], 99), 10)
for a, (nombre, col) in zip(ax, (("NSA", "logM_nsa"), ("Pipe3D", "log_mstar"))):
    yy = cohorte[col].to_numpy()[TRc] + 0.4 * MI[TRc]
    a.hexbin(GI[TRc], yy, gridsize=50, cmap="Greys", mincnt=1, norm=mcolors.LogNorm(vmin=0.3))
    a.plot(gx, TAYLOR[0] + TAYLOR[1] * gx, color=COLOR["Katachi"], lw=2, label="Taylor (2011)")
    f = coef[f"ajustada con masa {nombre}"]
    a.plot(gx, f["A"] + f["B"] * gx, color=COLOR["B1"], lw=2, ls="--", label=f"ajuste: A = {f['A']:.2f}, B = {f['B']:.2f}")
    a.set(xlabel="g − i [mag]", title=f"Masa {nombre}"); a.legend(fontsize=7)
ax[0].set_ylabel("log M* + 0.4 Mᵢ"); plt.suptitle("Relación color–M/L en entrenamiento (término −0.4 Mᵢ fijo)", y=1.02)
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** La relación color–masa es el método clásico para estimar masas desde fotometría. Ajustarla contra la masa del NSA y contra la de Pipe3D separa dos cosas: qué tan bien reproduce una estimación hecha con la misma fotometría y qué tan bien predice una medición independiente, que es lo que importa para el proyecto.

**Qué observamos.** _Se completa con los resultados de la corrida._

Ahora ajustamos las variantes de B1 para las tres propiedades y las comparamos con validación cruzada de 5 particiones dentro de entrenamiento, repetida con las tres semillas.

In [ ]:
VARIANTES = {"log_mstar": ["Taylor", "Lineal", "Árboles"], "log_sfr": ["Lineal", "Árboles"], "d4000": ["Lineal", "Árboles"]}
filas = []
for k, t in enumerate(OBJETIVOS):
    for v in VARIANTES[t]:
        errs = []
        for s in SEMILLAS:
            for a, b in KFold(5, shuffle=True, random_state=s).split(TRc):
                f = ajustar_b1(v, t, TRc[a], seed=s)
                errs.append(DG.metricas(Y[TRc[b], k], f(GI[TRc[b]], MI[TRc[b]]))["RMSE"])
        filas.append({"propiedad": CORTO[t], "variante": v, "RMSE CV": np.mean(errs),
                      "error estándar": np.std(errs, ddof=1) / np.sqrt(len(errs)), "parámetros": {"Taylor": 0, "Lineal": 2 if k == 0 else 3}.get(v, "—")})
CV_B1 = pd.DataFrame(filas)
ELECCION_B1 = {}
for t in OBJETIVOS:
    d = CV_B1[(CV_B1.propiedad == CORTO[t]) & (CV_B1.variante != "Taylor")].set_index("variante")
    limite = d["RMSE CV"].min() + d.loc[d["RMSE CV"].idxmin(), "error estándar"]
    ELECCION_B1[t] = "Lineal" if d.loc["Lineal", "RMSE CV"] <= limite else "Árboles"
CV_B1["elegida"] = [ELECCION_B1[t] == v for t, v in zip([k for k in OBJETIVOS for _ in VARIANTES[k]], CV_B1.variante)]
display(CV_B1.round(4))
print("Variante elegida por la regla de un error estándar:", {CORTO[t]: v for t, v in ELECCION_B1.items()})

B1F = {t: ajustar_b1(ELECCION_B1[t], t, TRc) for t in OBJETIVOS}
def predecir_b1(idx, gi=None, mi=None):
    P = np.full((len(cohorte), 3), np.nan)
    gi = GI[idx] if gi is None else gi
    mi = MI[idx] if mi is None else mi
    for k, t in enumerate(OBJETIVOS):
        P[idx, k] = B1F[t](gi, mi)
    return P
P_B1 = predecir_b1(np.where(cub)[0])

**Por qué lo hacemos.** La validación cruzada dentro de entrenamiento permite elegir la variante sin usar el conjunto de validación, que queda para medir el resultado. La regla de un error estándar prefiere el modelo más simple cuando la diferencia está dentro del ruido de la estimación.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 7.3 B2: red convolucional

| Elemento | Valor |
|---|---|
| Arquitectura | Tres ResNet50 en cadena: imagen → log M\*; imagen + plano de M\* → log SFR; imagen + planos de M\* y SFR → D4000 |
| Inicialización | Pesos de ImageNet; capa final Linear(2048 → 1); primera convolución nueva en las redes con 4 y 5 canales |
| Entrada | 300 × 300 píxeles; bandas z, r, g en el lugar de R, G, B |
| Pérdida y optimizador | MSE y Adam (tasa 10⁻³) por red; las predicciones previas no propagan gradiente |
| Programa | 30 épocas, tasa con calentamiento de 2 épocas y descenso en coseno, precisión mixta bf16, lote de 32 |
| Aumentación | Rotación aleatoria de 0 a 360° en cada época |
| Predicción | Promedio de las 8 rotaciones de 90° y reflejos de cada imagen (*test-time augmentation*) |
| Semillas | 42, 7 y 2026 |

La celda siguiente entrena cada semilla. Si un entrenamiento ya existe en Drive, se reutiliza; si quedó a medias, se reanuda desde su último punto de control. Con una A100, cada semilla tarda unos 20 minutos.

In [ ]:
MEJ = MejorasConfig(programa_rapido=True, epocas_rapido=EPOCAS, precision_mixta=True)
CFG = TrainConfig(fused_adam=DEVICE.type == "cuda", channels_last=DEVICE.type == "cuda", freeze_pretrained=False)
EXPS = [Experimento("base", MEJ, s, descripcion="Baseline B2") for s in SEMILLAS]
CTX = preparar_datos(cohorte, X300, NORM, fcfg, paths, guard, EXPS, tag=TAG)
t0 = time.time()
for e in EXPS:
    correr(e, CTX, paths, guard, DEVICE, base=CFG, max_epochs=EPOCAS if LIMITE else None,
           log=lambda s: print(s, flush=True))
print(f"Entrenamientos listos en {(time.time() - t0) / 60:.0f} min:", [e.identificador(fcfg.crop_px) for e in EXPS])

def cargar_red(semilla):
    e = Experimento("base", MEJ, semilla)
    t = Trainer(replace(CFG, seed=semilla), NORM, fcfg.out_px, paths.run_dir(e.identificador(fcfg.crop_px)), guard, DEVICE,
                log=lambda *_: None, mejoras=MEJ)
    t.load_final()
    return t

def datos(p):
    return GPUData(X300[IDX[p]], Y[IDX[p]].astype(np.float32), DEVICE)

P_B2 = {s: np.full((len(cohorte), 3), np.nan) for s in SEMILLAS}
REDES = {s: cargar_red(s) for s in SEMILLAS}
for p in ("Train", "Validation"):
    D = datos(p)
    for s, red in REDES.items():
        P_B2[s][IDX[p]] = red.predict(D, tta=True)
    del D
for s in SEMILLAS:
    pd.DataFrame({"mangaid": cohorte.mangaid, "particion": cohorte.particion,
                  **{f"pred_{t}": P_B2[s][:, k] for k, t in enumerate(OBJETIVOS)}}).query("particion != 'Test'") \
      .pipe(lambda d: guard.write(RES / f"predicciones_B2_seed{s}.csv", lambda f: d.to_csv(f, index=False), allow_replace=True))

### 7.4 La escalera en validación

Comparamos los modelos en las galaxias de validación. B1 solo cubre una parte, así que la tabla tiene dos bloques: todas las galaxias (piso y B2) y las cubiertas por B1 (los tres modelos sobre las mismas galaxias). Para B2 se muestra la semilla principal con su intervalo, y la media y la desviación entre las tres semillas.

In [ ]:
P_PISO = np.tile(Y[IDX["Train"]].mean(0), (len(cohorte), 1))
VA = IDX["Validation"]
bloques = []
for nombre_bloque, masc in (("todas", None), ("cubiertas por B1", cub)):
    mods = {"Piso": P_PISO, "B2": P_B2[SEMILLA]} if masc is None else {"Piso": P_PISO, "B1": P_B1, "B2": P_B2[SEMILLA]}
    tb = tabla(mods, VA, masc)
    sel = VA if masc is None else VA[masc[VA]]
    sem = {(CORTO[t]): [DG.metricas(Y[sel, k], P_B2[s][sel, k])["RMSE"] for s in SEMILLAS] for k, t in enumerate(OBJETIVOS)}
    tb["RMSE 3 semillas"] = [f"{np.mean(sem[r.propiedad]):.3f} ± {np.std(sem[r.propiedad], ddof=1):.3f}"
                             if r.modelo == "B2" and len(SEMILLAS) > 1 else "" for r in tb.itertuples()]
    tb.insert(0, "galaxias", nombre_bloque)
    bloques.append(tb)
ESCALERA = pd.concat(bloques, ignore_index=True)
display(ordenar(ESCALERA[["galaxias", "modelo", "propiedad", "N", "RMSE", "IC 95 %", "RMSE 3 semillas", "sesgo", "sigma_NMAD",
                          "catastroficos", "R2"]], ["galaxias", "propiedad", "modelo"]))

sel = VA[cub[VA]]
filas = []
for k, t in enumerate(OBJETIVOS):
    for s in SEMILLAS:
        d, lo, hi = DG.delta_rmse(Y[sel, k], P_B2[s][sel, k], P_B1[sel, k], n=N_BOOT)
        filas.append({"propiedad": CORTO[t], "semilla": s, "ΔRMSE (B2 − B1)": d, "IC 95 %": f"[{lo:+.3f}, {hi:+.3f}]",
                      "veredicto para B2": veredicto(lo, hi, MARGEN[t])})
display(pd.DataFrame(filas).set_index(["propiedad", "semilla"]).round(4))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
for a, t in zip(ax, OBJETIVOS):
    r = ESCALERA[(ESCALERA.galaxias == "cubiertas por B1") & (ESCALERA.propiedad == CORTO[t])]
    ic = np.array([[float(v) for v in s.strip("[]").split(",")] for s in r["IC 95 %"]])
    a.barh(r.modelo, r.RMSE, xerr=np.abs(ic.T - r.RMSE.to_numpy()), color=[COLOR[m] for m in r.modelo], capsize=3)
    a.axvline(RUIDO[t], color="k", ls=":", lw=1, label="error de la etiqueta")
    a.axvline(KATACHI_PUBLICADO[t], color=COLOR["Katachi"], ls="--", lw=1, label="Katachi publicado (prueba)")
    a.set(xlabel=f"RMSE en validación · {NOMBRES[t]}"); a.invert_yaxis()
ax[0].legend(fontsize=7, loc="lower right"); plt.suptitle("Escalera de modelos de referencia (galaxias cubiertas por B1)", y=1.03)
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** Es la comparación central del avance: cuánto se gana al pasar de no usar información, a usar dos números de fotometría, a usar la imagen completa. La diferencia pareada (ΔRMSE sobre las mismas galaxias) indica si la ventaja de un peldaño sobre otro es mayor que la incertidumbre.

**Qué observamos.** _Se completa con los resultados de la corrida._

## 8. Importancia de las características

En B1 las características son dos variables, y su importancia se lee en los coeficientes y con permutación. En B2 las "características" son las bandas, las regiones de la imagen y los planos que pasan de una red a otra. Usamos cuatro métodos, todos medidos sobre validación:

| Método | Tipo | Qué responde |
|---|---|---|
| Coeficientes y permutación en B1 | Embebido y modelo-agnóstico | ¿Cuánto aportan el color y la luminosidad? |
| Oclusión de bandas en B2 | Perturbación de la entrada (equivalente a la permutación) | ¿Qué banda usa la red para cada propiedad? |
| Oclusión radial en B2 | Perturbación de la entrada | ¿Qué región de la galaxia (núcleo o afueras) contiene la información? |
| Permutación de los planos de la cadena | Permutación | ¿Cuánto depende cada red de la predicción anterior? |
| Mapas SHAP (Lundberg y Lee 2017) | Atribución por píxel | ¿Qué píxeles suben o bajan cada predicción, y cómo cambia entre poblaciones? |

La importancia de cada elemento se mide como el **aumento del RMSE** al quitarle la información, con su intervalo por *bootstrap* pareado.

### 8.1 B1: color frente a luminosidad

In [ ]:
rng = np.random.default_rng(SEMILLA)
filas = []
base_va = predecir_b1(VAc)
for k, t in enumerate(OBJETIVOS):
    for var in ("g − i", "Mᵢ"):
        deltas = []
        for rep in range(30):
            pi = rng.permutation(len(VAc))
            gi, mi = GI[VAc].copy(), MI[VAc].copy()
            if var == "g − i":
                gi = gi[pi]
            else:
                mi = mi[pi]
            pp = B1F[t](gi, mi)
            deltas.append(DG.metricas(Y[VAc, k], pp)["RMSE"] - DG.metricas(Y[VAc, k], base_va[VAc, k])["RMSE"])
        filas.append({"propiedad": CORTO[t], "variable": var, "ΔRMSE": np.mean(deltas), "desv.": np.std(deltas)})
IMP_B1 = pd.DataFrame(filas)
lin = {t: (B1F[t].est if hasattr(B1F[t], "est") else None) for t in OBJETIVOS}
filas = []
for t in OBJETIVOS:
    e = lin[t]
    if isinstance(e, LinearRegression):
        sd = [np.std(GI[TRc])] + ([np.std(MI[TRc])] if t != "log_mstar" else [])
        b = list(e.coef_)
        if t == "log_mstar":
            b, sd = b + [-0.4], sd + [np.std(MI[TRc])]
        filas.append({"propiedad": CORTO[t], "coef. g − i": b[0], "coef. Mᵢ": b[1],
                      "efecto de 1σ de g − i": b[0] * sd[0], "efecto de 1σ de Mᵢ": b[1] * sd[1]})
display(pd.DataFrame(filas).set_index("propiedad").round(3))

fig, ax = plt.subplots(1, 3, figsize=(12, 2.6))
for a, t in zip(ax, OBJETIVOS):
    d = IMP_B1[IMP_B1.propiedad == CORTO[t]]
    a.barh(d.variable, d["ΔRMSE"], xerr=d["desv."], color=COLOR["B1"], capsize=3)
    a.set(xlabel=f"aumento del RMSE · {CORTO[t]}"); a.invert_yaxis()
plt.suptitle("B1: importancia por permutación en validación (30 repeticiones)", y=1.05); plt.tight_layout(); plt.show()

**Por qué lo hacemos.** En la relación color–masa, la luminosidad fija la escala de la masa y el color la corrige por la edad y el polvo de la población estelar. Medir cuánto empeora el modelo al romper cada variable indica qué parte de la información aporta cada una.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 8.2 B2: bandas y regiones de la imagen

Para ocluir una banda, la ponemos en el nivel del cielo (0 tras el estiramiento) y dejamos las otras dos intactas. Para ocluir una región, ponemos en el nivel del cielo las tres bandas dentro de un anillo, medido en unidades de R50 de cada galaxia. La oclusión produce imágenes que la red no vio al entrenar, así que mide **qué información usa** la red, no qué información existe.

In [ ]:
D_VA = datos("Validation")
r50_va = R50[VA]
ANILLOS = [(0, 0.5), (0.5, 1), (1, 2), (2, 4), (4, np.inf)]
ETQ_ANILLO = ["0–0.5", "0.5–1", "1–2", "2–4", "> 4"]
filas = []
for s, red in REDES.items():
    base = DG.predecir_con(red, D_VA)
    pruebas = [("banda", b, dict(ocluir_banda=i)) for i, b in enumerate(BANDAS_RED)]
    pruebas += [("anillo", e, dict(anillo=a, r50_px=r50_va)) for a, e in zip(ANILLOS, ETQ_ANILLO)]
    pruebas += [("plano", p, dict(permutar=p, seed=s)) for p in ("masa→SFR", "masa→D4000", "SFR→D4000")]
    for tipo, nombre, kw in pruebas:
        pm = DG.predecir_con(red, D_VA, **kw)
        for k, t in enumerate(OBJETIVOS):
            d, lo, hi = DG.delta_rmse(Y[VA, k], pm[:, k], base[:, k], n=N_BOOT)
            filas.append({"semilla": s, "tipo": tipo, "elemento": nombre, "propiedad": CORTO[t], "ΔRMSE": d, "lo": lo, "hi": hi})
IMP_B2 = pd.DataFrame(filas)
guard.write(RES / "importancia_B2.csv", lambda f: IMP_B2.to_csv(f, index=False), allow_replace=True)
resumen = IMP_B2.groupby(["tipo", "elemento", "propiedad"], sort=False)["ΔRMSE"].agg(["mean", "std"]).unstack("propiedad")
display(resumen.round(4))

def barras(tipo, titulo, etiquetas):
    fig, ax = plt.subplots(1, 3, figsize=(14, 2.9))
    for a, t in zip(ax, OBJETIVOS):
        d = IMP_B2[(IMP_B2.tipo == tipo) & (IMP_B2.propiedad == CORTO[t])]
        g = d.groupby("elemento", sort=False)["ΔRMSE"]
        principal = d[d.semilla == SEMILLA].set_index("elemento").loc[etiquetas]
        a.barh(etiquetas, g.mean().loc[etiquetas], color=COLOR["B2"], alpha=0.85)
        a.errorbar(principal["ΔRMSE"], etiquetas, xerr=[principal["ΔRMSE"] - principal.lo, principal.hi - principal["ΔRMSE"]],
                   fmt="none", ecolor="k", capsize=3)
        for s in SEMILLAS:
            v = d[d.semilla == s].set_index("elemento").loc[etiquetas, "ΔRMSE"]
            a.plot(v, etiquetas, "o", ms=3, color="#e9c46a")
        a.axvline(0, color="k", lw=0.6); a.set(xlabel=f"aumento del RMSE · {CORTO[t]}"); a.invert_yaxis()
    plt.suptitle(titulo + " (barra: media de las semillas; puntos: cada semilla; línea: IC 95 % de la semilla principal)", y=1.05)
    plt.tight_layout(); plt.show()

barras("banda", "B2: oclusión de cada banda", BANDAS_RED)

fig, ax = plt.subplots(1, 3, figsize=(14, 3))
for a, t in zip(ax, OBJETIVOS):
    d = IMP_B2[(IMP_B2.tipo == "anillo") & (IMP_B2.propiedad == CORTO[t])]
    g = d.groupby("elemento", sort=False)["ΔRMSE"]
    a.errorbar(range(len(ETQ_ANILLO)), g.mean().loc[ETQ_ANILLO], yerr=g.std().loc[ETQ_ANILLO].fillna(0), marker="o",
               color=COLOR["B2"], capsize=3)
    a.axhline(0, color="k", lw=0.6)
    a.set(xticks=range(len(ETQ_ANILLO)), xticklabels=ETQ_ANILLO, xlabel="anillo ocluido [r / R50]", ylabel=f"aumento del RMSE · {CORTO[t]}")
plt.suptitle("B2: oclusión radial (media ± desviación entre semillas)", y=1.03); plt.tight_layout(); plt.show()

**Por qué lo hacemos.** La red usa la imagen completa, pero no sabemos qué parte de ella aprovecha. Si quitar la banda z o las afueras de la galaxia no cambia el error, la red no usa esa información. Las bandas indican si la red aprendió el color, y los anillos si se fija en el núcleo (donde dominan las estrellas viejas del bulbo) o en el disco (donde se forman las estrellas jóvenes).

**Qué observamos.** _Se completa con los resultados de la corrida._

### 8.3 B2: la cadena

Mezclamos entre galaxias la predicción de masa que recibe la red de SFR, y la de masa o la de SFR que recibe la red de D4000. La mezcla conserva la distribución de los valores pero rompe su relación con cada imagen. Si el error no cambia, la red no usa el plano.

In [ ]:
barras("plano", "B2: permutación de los planos de la cadena", ["masa→SFR", "masa→D4000", "SFR→D4000"])

**Por qué lo hacemos.** La cadena es la decisión de diseño más particular de Katachi: supone que conocer la masa ayuda a estimar la SFR y la edad. Esta prueba mide si la red realmente usa esa información o si obtiene lo mismo de la imagen.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 8.4 B2: mapas SHAP

SHAP asigna a cada píxel su contribución a la predicción, en las unidades de la propiedad, respecto a la predicción media de un conjunto de referencia (32 galaxias de entrenamiento). Usamos `GradientExplainer` sobre la cadena completa, de modo que la contribución incluye el camino por las redes anteriores. Calculamos los mapas para 48 galaxias de validación, la mitad formadoras y la mitad apagadas, con la semilla principal.

Mostramos tres galaxias de ejemplo y el **perfil radial medio** de cada población: la contribución promedio por píxel en anillos de r / R50.

In [ ]:
red = REDES[SEMILLA]
rng = np.random.default_rng(SEMILLA)
pob_va = cohorte.poblacion.to_numpy()[VA]
n_pob = N_SHAP // 2
sel = np.concatenate([rng.choice(np.where(pob_va == p)[0], min(n_pob, int((pob_va == p).sum())), replace=False)
                      for p in ("formadora", "apagada")])
fondo = DG.entrada_red(red, torch.from_numpy(X300[rng.choice(IDX["Train"], 32, replace=False)].astype(np.float32))).float()
xs = DG.entrada_red(red, torch.from_numpy(X300[VA[sel]].astype(np.float32))).float()
t0 = time.time()
SHAP = {}
for k, t in enumerate(OBJETIVOS):
    modelo = ChainTarget(red.model, k).eval()
    SHAP[t] = SM.shap_maps(modelo, fondo, xs, batch=8, nsamples=SHAP_MUESTRAS).sum(-1)      # (N, H, W)
print(f"Mapas SHAP calculados en {(time.time() - t0) / 60:.1f} min")

bordes_u = np.arange(0, 4.01, 0.25)
lado = xs.shape[-1]
yy, xx = np.indices((lado, lado)); rr = np.hypot(yy - (lado - 1) / 2, xx - (lado - 1) / 2)
def perfil(mapa, r50):
    u = rr / r50
    return np.array([mapa[(u >= a) & (u < b)].mean() if ((u >= a) & (u < b)).any() else np.nan
                     for a, b in zip(bordes_u[:-1], bordes_u[1:])])

g = cohorte.iloc[VA[sel]]
gm, gp = g.log_mstar.to_numpy(), g.poblacion.to_numpy()
e1 = int(np.where(gp == "formadora")[0][np.argmin(gm[gp == "formadora"])])
e3 = int(np.where(gp == "apagada")[0][np.argmax(gm[gp == "apagada"])])
resto = [j for j in np.argsort(np.abs(gm - np.median(gm))) if j not in (e1, e3)]
ejemplos = [e1, int(resto[0]), e3]
fig, ax = plt.subplots(3, 4, figsize=(12, 9))
for fila_, j in enumerate(ejemplos):
    f = g.iloc[j]
    ax[fila_, 0].imshow(DG.vista_rgb(xs[j].cpu().numpy()), origin="lower")
    ax[fila_, 0].set_title(f"{f.poblacion} · log M* {f.log_mstar:.1f}\nSFR {f.log_sfr:.1f} · D4000 {f.d4000:.2f}", fontsize=8)
    for c, t in enumerate(OBJETIVOS, 1):
        m = ndimage.gaussian_filter(SHAP[t][j], 2)
        v = np.percentile(np.abs(m), 99.5)
        im = ax[fila_, c].imshow(m, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
        ax[fila_, c].add_patch(plt.Circle(((lado - 1) / 2, (lado - 1) / 2), f.R50_px, fill=False, color="k", lw=0.6, ls="--"))
        ax[fila_, c].set_title(f"SHAP {CORTO[t]} (rojo: sube la predicción)", fontsize=8)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(14, 3.3))
uc = (bordes_u[:-1] + bordes_u[1:]) / 2
for a, t in zip(ax, OBJETIVOS):
    for p, c in (("formadora", "#457b9d"), ("apagada", "#e63946")):
        js = np.where(g.poblacion.to_numpy() == p)[0]
        prof = np.array([perfil(SHAP[t][j], g.R50_px.iloc[j]) for j in js])
        mu, se = np.nanmean(prof, 0), np.nanstd(prof, 0) / np.sqrt(np.sum(np.isfinite(prof), 0))
        a.plot(uc, mu, color=c, label=p); a.fill_between(uc, mu - se, mu + se, color=c, alpha=0.25)
    a.axhline(0, color="k", lw=0.6); a.set(xlabel="r / R50", ylabel="SHAP medio por píxel", title=CORTO[t])
ax[0].legend(fontsize=8); plt.suptitle("Perfil radial de las contribuciones SHAP por población (media ± error estándar)", y=1.03)
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** La oclusión indica qué región importa en promedio; SHAP indica en qué dirección empuja cada píxel y si la red razona distinto en una galaxia formadora y en una apagada. Es la forma estándar de interpretar redes de este tipo en astronomía, y la que usó Katachi para relacionar la morfología con la historia de formación estelar.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 8.5 Síntesis de la importancia

_Se completa con los resultados de la corrida: tabla que resume, para cada propiedad, qué información usa cada modelo y su interpretación física._

## 9. Sub y sobreajuste

Un modelo **subajusta** cuando su error es alto tanto en entrenamiento como en validación: le falta capacidad o información. **Sobreajusta** cuando su error de entrenamiento es mucho menor que el de validación: memoriza particularidades de las galaxias de entrenamiento (Géron 2022, cap. 4). En este problema hay una referencia adicional: un error de entrenamiento menor que el error de las propias etiquetas indica que el modelo aprendió también su ruido.

### 9.1 Entrenamiento frente a validación

In [ ]:
filas = []
for nombre, P, masc in (("B1", P_B1, cub), *((f"B2 semilla {s}", P_B2[s], None) for s in SEMILLAS)):
    for k, t in enumerate(OBJETIVOS):
        f = {"modelo": nombre, "propiedad": CORTO[t]}
        for part in ("Train", "Validation"):
            sel = IDX[part] if masc is None else IDX[part][masc[IDX[part]]]
            m = evaluar(Y[sel, k], P[sel, k], t)
            f.update({f"RMSE {part}": m["RMSE"], f"sesgo {part}": m["sesgo"], f"σ_NMAD {part}": m["sigma_NMAD"]})
        if nombre == "B1":
            f["RMSE CV"] = CV_B1[(CV_B1.propiedad == CORTO[t]) & (CV_B1.variante == ELECCION_B1[t])]["RMSE CV"].iloc[0]
        f["cociente Validation / Train"] = f["RMSE Validation"] / f["RMSE Train"]
        f["error de la etiqueta"] = RUIDO[t]
        filas.append(f)
AJUSTE = ordenar(pd.DataFrame(filas), ["propiedad", "modelo"])
display(AJUSTE.round(3))

**Por qué lo hacemos.** Es la comparación directa que define el sub o sobreajuste. El cociente entre el error de validación y el de entrenamiento resume la brecha: cerca de 1 indica que el modelo generaliza lo que aprendió, y muy por encima de 1 que memoriza. Comparar el error de entrenamiento con el de la etiqueta indica si el modelo ya ajusta su ruido.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 9.2 Curvas de aprendizaje de B1

Entrenamos B1 con fracciones crecientes del conjunto de entrenamiento y medimos el error en esas mismas galaxias y en la partición de validación cruzada que se dejó fuera. Mostramos las dos variantes, la lineal y la de árboles, porque se comportan de forma opuesta.

In [ ]:
fracciones = np.array([0.05, 0.1, 0.2, 0.4, 0.7, 1.0])
filas = []
for k, t in enumerate(OBJETIVOS):
    for v in ("Lineal", "Árboles"):
        for s in SEMILLAS:
            for a, b in KFold(5, shuffle=True, random_state=s).split(TRc):
                orden = np.random.default_rng(s).permutation(a)
                for fr in fracciones:
                    sub = TRc[orden[:max(10, int(fr * len(orden)))]]
                    f = ajustar_b1(v, t, sub, seed=s)
                    filas.append({"propiedad": t, "variante": v, "n": len(sub),
                                  "entrenamiento": DG.metricas(Y[sub, k], f(GI[sub], MI[sub]))["RMSE"],
                                  "validación cruzada": DG.metricas(Y[TRc[b], k], f(GI[TRc[b]], MI[TRc[b]]))["RMSE"]})
CURVA = pd.DataFrame(filas)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, t in zip(ax, OBJETIVOS):
    for v, ls in (("Lineal", "-"), ("Árboles", "--")):
        d = CURVA[(CURVA.propiedad == t) & (CURVA.variante == v)].groupby("n")
        for col, c in (("entrenamiento", "#e9c46a"), ("validación cruzada", COLOR["B1"])):
            mu, sd = d[col].mean(), d[col].std()
            a.plot(mu.index, mu, ls, color=c, marker="o", ms=3, label=f"{v} · {col}")
            a.fill_between(mu.index, mu - sd, mu + sd, color=c, alpha=0.15)
    a.axhline(RUIDO[t], color="k", ls=":", lw=1, label="error de la etiqueta")
    a.set(xscale="log", xlabel="galaxias de entrenamiento", ylabel="RMSE", title=NOMBRES[t])
ax[0].legend(fontsize=6.5); plt.suptitle("Curvas de aprendizaje de B1 (media ± desviación en 5 particiones × 3 semillas)", y=1.03)
plt.tight_layout(); plt.show()

**Por qué lo hacemos.** Las curvas de aprendizaje indican si más datos ayudarían. Si las dos curvas convergen a un error alto, el modelo subajusta y más galaxias no lo mejorarán: hace falta más información o más capacidad. Si quedan separadas, el modelo sobreajusta y más datos o más regularización reducirían la brecha.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 9.3 Curvas de entrenamiento de B2

En cada época se registra el error de entrenamiento (con la aumentación activa) y el de validación de cada una de las tres redes. Entrenar la red con distintas cantidades de galaxias costaría horas de GPU, así que para B2 la evolución por época y la comparación de la sección 9.1 cumplen el papel de la curva de aprendizaje.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, (t, n) in zip(ax, zip(OBJETIVOS, ("mstar", "sfr", "d4000"))):
    for s in SEMILLAS:
        h = pd.read_csv(paths.run_dir(Experimento("base", MEJ, s).identificador(fcfg.crop_px)) / "history.csv")
        a.plot(h.epoch, np.sqrt(h[f"train_loss_{n}"]), color="#e9c46a", alpha=0.8, label="entrenamiento" if s == SEMILLA else None)
        a.plot(h.epoch, np.sqrt(h[f"val_loss_{n}"]), color=COLOR["B2"], alpha=0.8, label="validación" if s == SEMILLA else None)
    k = OBJETIVOS.index(t)
    a.axhline(np.mean([DG.metricas(Y[IDX["Train"], k], P_B2[s][IDX["Train"], k])["RMSE"] for s in SEMILLAS]),
              color="#e9c46a", ls="--", lw=1, label="entrenamiento sin aumentación, al final")
    a.axhline(RUIDO[t], color="k", ls=":", lw=1, label="error de la etiqueta")
    a.set(yscale="log", xlabel="época", ylabel="RMSE", title=NOMBRES[t])
ax[0].legend(fontsize=7); plt.suptitle("B2: error por época (una línea por semilla)", y=1.03); plt.tight_layout(); plt.show()

**Por qué lo hacemos.** La forma de las curvas indica en qué momento la red deja de generalizar. Si la curva de validación sube mientras la de entrenamiento sigue bajando, la red empieza a memorizar. La línea discontinua muestra el error de entrenamiento medido igual que el de validación (sin aumentación), que es el comparable.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 9.4 Residuos

Los residuos (predicción − etiqueta) muestran patrones que una métrica global no ve. Graficamos la predicción frente a la etiqueta y la mediana del residuo, con su banda del 16 al 84 %, frente al valor real y frente al redshift. Una pendiente negativa frente al valor real indica **regresión a la media**: el modelo sobreestima los valores bajos y subestima los altos, una señal de subajuste.

In [ ]:
def densidad(a, y, p, t, titulo):
    lim = np.nanpercentile(np.r_[y, p], [0.5, 99.5])
    a.hexbin(y, p, gridsize=45, cmap="viridis", mincnt=1, bins="log", extent=(*lim, *lim))
    a.plot(lim, lim, "w-", lw=1)
    if t != "d4000":
        for d in (-UMBRAL_CATASTROFICO, UMBRAL_CATASTROFICO):
            a.plot(lim, lim + d, "w:", lw=0.8)
    m = evaluar(y, p, t)
    texto = f"RMSE {m['RMSE']:.3f}\nsesgo {m['sesgo']:+.3f}\nσ_NMAD {m['sigma_NMAD']:.3f}"
    if "catastroficos" in m:
        texto += f"\n> 0.3 dex: {100 * m['catastroficos']:.1f} %"
    a.text(0.03, 0.97, texto, transform=a.transAxes, va="top", fontsize=7, color="w")
    a.set(xlim=lim, ylim=lim, xlabel=f"{NOMBRES[t]} (Pipe3D)", ylabel="predicción", title=titulo)

fig, ax = plt.subplots(2, 3, figsize=(14, 9))
sel1 = VA[cub[VA]]
for c, t in enumerate(OBJETIVOS):
    densidad(ax[0, c], Y[sel1, c], P_B1[sel1, c], t, f"B1 · {CORTO[t]} (validación)")
    densidad(ax[1, c], Y[VA, c], P_B2[SEMILLA][VA, c], t, f"B2 · {CORTO[t]} (validación)")
plt.tight_layout(); plt.show()

z = cohorte.redshift.to_numpy()
fig, ax = plt.subplots(2, 3, figsize=(15, 6.5))
for c, t in enumerate(OBJETIVOS):
    for nombre, P, sel_ in (("B1", P_B1, sel1), ("B2", P_B2[SEMILLA], VA)):
        r = P[sel_, c] - Y[sel_, c]
        for fila_, x, bordes in ((0, Y[sel_, c], np.quantile(Y[VA, c], np.linspace(0, 1, 11))),
                                 (1, z[sel_], np.quantile(z[VA], np.linspace(0, 1, 11)))):
            mm = DG.mediana_movil(x, r, bordes)
            ax[fila_, c].plot(mm["x"], mm["mediana"], "o-", ms=3, color=COLOR[nombre], label=nombre)
            ax[fila_, c].fill_between(mm["x"], mm["p16"], mm["p84"], color=COLOR[nombre], alpha=0.15)
    ax[0, c].set(xlabel=f"{NOMBRES[t]} (Pipe3D)", ylabel="residuo", title=CORTO[t])
    ax[1, c].set(xlabel="redshift", ylabel="residuo")
    if any(len(l.get_xdata()) for l in ax[1, c].lines) and np.all(z[VA] > 0):
        ax[1, c].set_xscale("log")
    for a in ax[:, c]:
        a.axhline(0, color="k", lw=0.6)
ax[0, 0].legend(fontsize=8); plt.suptitle("Mediana del residuo y banda 16–84 % en validación", y=1.01); plt.tight_layout(); plt.show()

filas = []
for nombre, P, sel_ in (("B1", P_B1, sel1), *((f"B2 semilla {s}", P_B2[s], VA) for s in SEMILLAS)):
    for c, t in enumerate(OBJETIVOS):
        r = P[sel_, c] - Y[sel_, c]
        filas.append({"modelo": nombre, "propiedad": CORTO[t], "pendiente residuo–valor real": np.polyfit(Y[sel_, c], r, 1)[0],
                      **{f"sesgo {p}": r[cohorte.poblacion.to_numpy()[sel_] == p].mean() for p in ("formadora", "apagada")}})
display(ordenar(pd.DataFrame(filas), ["propiedad", "modelo"]).round(3))

**Por qué lo hacemos.** Los residuos revelan dónde y por qué falla cada modelo. El residuo frente al redshift es especialmente relevante para B2: una imagen no indica a qué distancia está la galaxia, y una galaxia pequeña y cercana puede verse igual que una grande y lejana.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 9.5 Diagnóstico

_Se completa con los resultados de la corrida: tabla con el diagnóstico de cada modelo y propiedad (subajuste, buen ajuste o sobreajuste) y la evidencia de cada señal._

## 10. Evaluación en el conjunto de prueba

Con los modelos, las variantes y los criterios fijados, registramos la elección y abrimos el conjunto de prueba **una sola vez**. El registro guarda la fecha, el código y una huella de las predicciones de validación, de modo que cualquier cambio posterior sea detectable.

In [ ]:
huella_val = hashlib.sha256(np.round(np.stack([P_B2[s][VA] for s in SEMILLAS]), 6).tobytes()).hexdigest()
registro = {"fecha_utc": datetime.now(timezone.utc).isoformat(), "commit": COMMIT, "B1": ELECCION_B1,
            "B2": [e.identificador(fcfg.crop_px) for e in EXPS], "semilla_principal": SEMILLA,
            "criterios": "criterios.json", "huella_predicciones_validacion": huella_val}
f_reg = RES / "registro_prueba.json"
if f_reg.exists():
    previo = json.loads(f_reg.read_text())
    print("La prueba se abrió por primera vez el", previo["fecha_utc"])
    if previo["huella_predicciones_validacion"] != huella_val or previo["B1"] != ELECCION_B1:
        print("AVISO: los modelos cambiaron desde la primera apertura; esta evaluación no es una prueba independiente.")
else:
    guard.write_json(f_reg, registro)
    print("Elección registrada antes de abrir la prueba:", f_reg)

TE = IDX["Test"]
D_TE = datos("Test")
for s, red in REDES.items():
    P_B2[s][TE] = red.predict(D_TE, tta=True)
del D_TE
P_KAT = cohorte[["pred_mstar", "pred_sfr", "pred_d4000"]].to_numpy(float)
KAT = np.isfinite(P_KAT).all(1)                  # galaxias con predicción publicada de Katachi
for s in SEMILLAS:
    guard.write(RES / f"predicciones_prueba_B2_seed{s}.csv",
                lambda f, s=s: pd.DataFrame({"mangaid": cohorte.mangaid.iloc[TE],
                                             **{f"pred_{t}": P_B2[s][TE, k] for k, t in enumerate(OBJETIVOS)}}).to_csv(f, index=False),
                allow_replace=True)
print("Galaxias de prueba:", len(TE), "· con predicción de Katachi:", int(np.isfinite(P_KAT[TE]).all(1).sum()),
      "· cubiertas por B1:", int(cub[TE].sum()))

### 10.1 Métricas en prueba

In [ ]:
bloques = []
for nombre_bloque, masc, mods in (("todas", KAT, {"Piso": P_PISO, "Katachi": P_KAT, "B2": P_B2[SEMILLA]}),
                                  ("cubiertas por B1", cub & KAT, {"Piso": P_PISO, "B1": P_B1, "Katachi": P_KAT, "B2": P_B2[SEMILLA]})):
    tb = tabla(mods, TE, masc)
    sel_ = TE[masc[TE]]
    sem = {CORTO[t]: [DG.metricas(Y[sel_, k], P_B2[s][sel_, k])["RMSE"] for s in SEMILLAS] for k, t in enumerate(OBJETIVOS)}
    tb["RMSE 3 semillas"] = [f"{np.mean(sem[r.propiedad]):.3f} ± {np.std(sem[r.propiedad], ddof=1):.3f}"
                             if r.modelo == "B2" and len(SEMILLAS) > 1 else "" for r in tb.itertuples()]
    tb.insert(0, "galaxias", nombre_bloque)
    bloques.append(tb)
PRUEBA = pd.concat(bloques, ignore_index=True)
guard.write(RES / "metricas_prueba.csv", lambda f: PRUEBA.to_csv(f, index=False), allow_replace=True)
display(ordenar(PRUEBA[["galaxias", "modelo", "propiedad", "N", "RMSE", "IC 95 %", "RMSE 3 semillas", "sesgo", "sigma_NMAD",
                        "catastroficos", "R2"]], ["galaxias", "propiedad", "modelo"]))

TEK = TE[KAT[TE]]
fig, ax = plt.subplots(2, 3, figsize=(14, 9))
for c, t in enumerate(OBJETIVOS):
    densidad(ax[0, c], Y[TEK, c], P_KAT[TEK, c], t, f"Katachi (imágenes SDSS) · {CORTO[t]}")
    densidad(ax[1, c], Y[TEK, c], P_B2[SEMILLA][TEK, c], t, f"B2 (imágenes DESI) · {CORTO[t]}")
plt.suptitle("Predicción frente a etiqueta en las galaxias de prueba", y=1.0); plt.tight_layout(); plt.show()

**Por qué lo hacemos.** Es la medición final de los dos modelos de referencia, en galaxias que no intervinieron en ninguna decisión. La comparación visual con Katachi, sobre las mismas galaxias, muestra si las diferencias están en el núcleo de la distribución o en los extremos.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 10.2 ¿Reproduce el modelo la física de la muestra?

Un modelo puede tener un RMSE bajo y aun así deformar las relaciones que el catálogo debe servir para estudiar. Comparamos el plano SFR–M\* y la distribución del D4000 de las etiquetas con los que producen las predicciones.

In [ ]:
def contornos(a, x, y, color, etiqueta, rango):
    H, xe, ye = np.histogram2d(x, y, bins=40, range=rango)
    H = ndimage.gaussian_filter(H.T, 1.2)
    orden = np.sort(H.ravel())[::-1]
    acum = np.cumsum(orden) / orden.sum()
    niveles = sorted({orden[np.searchsorted(acum, q)] for q in (0.9, 0.68, 0.38)})
    a.contour((xe[:-1] + xe[1:]) / 2, (ye[:-1] + ye[1:]) / 2, H, levels=niveles, colors=color, linewidths=1.2)
    a.plot([], [], color=color, label=etiqueta)

rango = [[8.5, 11.8], [-3.5, 1.5]]
fig, ax = plt.subplots(1, 4, figsize=(17, 3.9), gridspec_kw={"width_ratios": [1, 1, 1, 1.1]})
for a, (nombre, P, c) in zip(ax[:3], (("Pipe3D (etiquetas)", np.c_[Y], "k"), ("Katachi", P_KAT, COLOR["Katachi"]),
                                      ("B2", P_B2[SEMILLA], COLOR["B2"]))):
    contornos(a, Y[TEK, 0], Y[TEK, 1], "0.75", "Pipe3D", rango)
    if nombre != "Pipe3D (etiquetas)":
        contornos(a, P[TEK, 0], P[TEK, 1], c, nombre, rango)
    a.plot(np.array(rango[0]), np.array(rango[0]) + CORTE_SSFR, "k--", lw=0.8)
    a.set(xlim=rango[0], ylim=rango[1], xlabel=NOMBRES["log_mstar"], ylabel=NOMBRES["log_sfr"], title=nombre); a.legend(fontsize=7, loc="lower right")
b = np.linspace(1.0, 2.3, 40)
ax[3].hist(Y[TEK, 2], b, color="0.8", label="Pipe3D")
ax[3].hist(P_KAT[TEK, 2], b, histtype="step", lw=1.6, color=COLOR["Katachi"], label="Katachi")
ax[3].hist(P_B2[SEMILLA][TEK, 2], b, histtype="step", lw=1.6, color=COLOR["B2"], label="B2")
ax[3].set(xlabel="D4000", ylabel="galaxias", title="Distribución del D4000"); ax[3].legend(fontsize=7)
plt.suptitle("Galaxias de prueba: contornos que encierran el 38, 68 y 90 % de la muestra", y=1.03); plt.tight_layout(); plt.show()

filas = []
for nombre, P in (("Pipe3D", np.c_[Y]), ("Katachi", P_KAT), ("B2", P_B2[SEMILLA])):
    ss = P[TEK, 1] - P[TEK, 0]
    filas.append({"": nombre, "fracción apagada (sSFR < −10.8)": np.mean(ss < CORTE_SSFR),
                  "D4000 > 1.6": np.mean(P[TEK, 2] > 1.6), "log SFR mínimo (percentil 2)": np.percentile(P[TEK, 1], 2)})
display(pd.DataFrame(filas).set_index("").round(3))

**Por qué lo hacemos.** Las dos poblaciones de galaxias y la separación entre ellas son el resultado físico más importante que el catálogo debe conservar. Si un modelo junta las galaxias apagadas con las formadoras, cualquier estudio de cómo se apaga la formación estelar quedaría sesgado, aunque el RMSE fuera aceptable.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 10.3 Comparación pareada con Katachi y desempeño por grupos

In [ ]:
pobl = cohorte.poblacion.to_numpy()
filas = []
for grupo, msk in (("todas", np.ones(len(cohorte), bool)), ("formadoras", pobl == "formadora"), ("apagadas", pobl == "apagada")):
    sel_ = TE[msk[TE] & KAT[TE]]
    for k, t in enumerate(OBJETIVOS):
        for s in SEMILLAS:
            d, lo, hi = DG.delta_rmse(Y[sel_, k], P_B2[s][sel_, k], P_KAT[sel_, k], n=N_BOOT)
            filas.append({"grupo": grupo, "propiedad": CORTO[t], "semilla": s, "N": len(sel_), "ΔRMSE (B2 − Katachi)": d,
                          "lo": lo, "hi": hi, "veredicto para B2": veredicto(lo, hi, MARGEN[t])})
PAR = pd.DataFrame(filas)
display(PAR.assign(**{"IC 95 %": [f"[{a:+.3f}, {b:+.3f}]" for a, b in zip(PAR.lo, PAR.hi)]})
           .drop(columns=["lo", "hi"]).set_index(["grupo", "propiedad", "semilla"]).round(4))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.4), sharey=True)
grupos = ["todas", "formadoras", "apagadas"]
for a, t in zip(ax, OBJETIVOS):
    for gi_, gr in enumerate(grupos):
        d = PAR[(PAR.grupo == gr) & (PAR.propiedad == CORTO[t])]
        for j, (_, r) in enumerate(d.iterrows()):
            yv, v = gi_ + (j - 1) * 0.18, r["ΔRMSE (B2 − Katachi)"]
            a.errorbar(v, yv, xerr=[[v - r.lo], [r.hi - v]], fmt="o", color=COLOR["B2"], capsize=3, ms=4)
    a.axvline(0, color="k", lw=0.8); a.axvspan(-MARGEN[t], MARGEN[t], color="0.9", zorder=0)
    a.set(xlabel=f"ΔRMSE (B2 − Katachi) · {CORTO[t]}", yticks=range(3), yticklabels=grupos)
ax[0].invert_yaxis()
plt.suptitle("B2 frente a Katachi en las mismas galaxias de prueba (tres semillas; banda gris: margen de equivalencia; negativo = B2 mejor)", y=1.04)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
bm = pd.cut(cohorte.log_mstar, BORDES_M).astype(str).to_numpy()
cats = [str(c) for c in pd.cut(cohorte.log_mstar, BORDES_M).cat.categories]
for a, (k, t) in zip(ax, enumerate(OBJETIVOS)):
    for j, (nombre, P, msk) in enumerate((("B1", P_B1, cub), ("Katachi", P_KAT, None), ("B2", P_B2[SEMILLA], None))):
        xs_, ys_, es_ = [], [], []
        for i, cat in enumerate(cats):
            sel_ = TE[(bm[TE] == cat) & KAT[TE] & (np.ones(len(cohorte), bool) if msk is None else msk)[TE]]
            sel_ = sel_[np.isfinite(P[sel_, k])]
            if len(sel_) >= 10:
                lo, hi = DG.rmse_ic(Y[sel_, k], P[sel_, k], n=N_BOOT)
                v = DG.metricas(Y[sel_, k], P[sel_, k])["RMSE"]
                xs_.append(i + (j - 1) * 0.2); ys_.append(v); es_.append([v - lo, hi - v])
        a.errorbar(xs_, ys_, yerr=np.array(es_).T, fmt="o-", color=COLOR[nombre], capsize=3, label=nombre, ms=4)
    a.set(xticks=range(len(cats)), xticklabels=cats, xlabel="intervalo de log M*", ylabel="RMSE", title=NOMBRES[t])
ax[0].legend(fontsize=7); plt.suptitle("Error por intervalo de masa en las galaxias de prueba (IC 95 %)", y=1.03); plt.tight_layout(); plt.show()

**Por qué lo hacemos.** Comparar sobre las mismas galaxias elimina la diferencia de muestras, que era la principal fuente de confusión al comparar cifras globales de conjuntos distintos. Separar por población y por masa muestra dónde gana o pierde cada modelo.

**Qué observamos.** _Se completa con los resultados de la corrida._

### 10.4 ¿Se alcanza el desempeño mínimo?

In [ ]:
filas = []
for k, t in enumerate(OBJETIVOS):
    for nombre, P, msk in (("B1", P_B1, cub), ("B2", P_B2[SEMILLA], None)):
        sel_ = TE if msk is None else TE[msk[TE]]
        lo, hi = DG.rmse_ic(Y[sel_, k], P[sel_, k], n=N_BOOT)
        piso = DG.metricas(Y[sel_, k], P_PISO[sel_, k])["RMSE"]
        filas.append({"criterio": "Viabilidad", "modelo": nombre, "propiedad": CORTO[t],
                      "valor": f"IC [{lo:.3f}, {hi:.3f}] frente al piso {piso:.3f}", "cumple": hi < piso})
    d = PAR[(PAR.grupo == "todas") & (PAR.propiedad == CORTO[t])]
    filas.append({"criterio": "Desempeño mínimo (no peor que Katachi)", "modelo": "B2", "propiedad": CORTO[t],
                  "valor": "límite superior de ΔRMSE: " + ", ".join(f"{h:+.3f}" for h in d.hi) + f" (margen {MARGEN[t]})",
                  "cumple": bool((d.hi < MARGEN[t]).all())})
rm = [DG.metricas(Y[TE, 0], P_B2[s][TE, 0])["RMSE"] for s in SEMILLAS]
filas.append({"criterio": "Deseable en masa", "modelo": "B2", "propiedad": "M*",
              "valor": f"RMSE {np.mean(rm):.3f} frente a {MASA_DESEABLE}", "cumple": np.mean(rm) <= MASA_DESEABLE})
VERIF = pd.DataFrame(filas)
display(VERIF.style.map(lambda v: "background-color: #d8f3dc" if v is True else ("background-color: #ffd6d6" if v is False else ""),
                        subset=["cumple"]))

**Qué observamos.** _Se completa con los resultados de la corrida._

### 10.5 Marco de referencia para los modelos siguientes

La tabla fija lo que un modelo nuevo deberá demostrar para considerarse una mejora sobre el baseline. Se guarda en `marco_referencia.csv`.

**Regla de decisión.** Un modelo nuevo mejora al baseline en una propiedad si, en las galaxias de validación y con tres semillas, el intervalo del 95 % de ΔRMSE = RMSE(nuevo) − RMSE(B2) queda completamente por debajo de 0 y ninguna otra propiedad empeora. Es equivalente si el intervalo cae dentro del margen. El conjunto de prueba solo se usa para confirmar al modelo final, una vez.

In [ ]:
filas = []
for k, t in enumerate(OBJETIVOS):
    rv = [DG.metricas(Y[VA, k], P_B2[s][VA, k])["RMSE"] for s in SEMILLAS]
    rt = [DG.metricas(Y[TE, k], P_B2[s][TE, k])["RMSE"] for s in SEMILLAS]
    pobs = {p: np.mean([DG.metricas(Y[VA[pobl[VA] == p], k], P_B2[s][VA[pobl[VA] == p], k])["RMSE"] for s in SEMILLAS])
            for p in ("formadora", "apagada")}
    filas.append({"propiedad": CORTO[t], "B2 validación (media 3 semillas)": np.mean(rv), "B2 prueba (media 3 semillas)": np.mean(rt),
                  "B2 validación, formadoras": pobs["formadora"], "B2 validación, apagadas": pobs["apagada"],
                  "variación entre semillas (validación)": np.std(rv, ddof=1) if len(rv) > 1 else np.nan,
                  "margen de equivalencia": MARGEN[t], "piso": SIGMA[t], "error de la etiqueta": RUIDO[t]})
MARCO = pd.DataFrame(filas).set_index("propiedad")
guard.write(RES / "marco_referencia.csv", lambda f: MARCO.to_csv(f), allow_replace=True)
display(MARCO.round(3))

## 11. Conclusiones de la fase de ingeniería del modelo (CRISP-ML(Q))

En CRISP-ML(Q), la fase de *Model Engineering* exige definir las métricas de calidad del modelo, documentar los riesgos que pueden comprometer su desempeño y aplicar métodos de aseguramiento de calidad, entre ellos la validación del desempeño medio con distintas semillas y el registro de los metadatos del entrenamiento (Studer et al. 2021).

| Tarea | Requisito | Riesgo | Control de calidad aplicado |
|---|---|---|---|
| **Definir la métrica** | Medida alineada con el uso científico del catálogo | Una métrica global que oculte errores en poblaciones pequeñas | RMSE en dex con su lectura como factor; sesgo, σ_NMAD y errores catastróficos; métricas por población y por masa (secciones 5 y 10.3) |
| **Fijar el desempeño mínimo** | Umbral con referencia externa | Elegir el umbral después de ver los resultados | Referencias históricas y criterios registrados antes de evaluar (sección 6) |
| **Seleccionar el modelo** | Algoritmo apropiado para el tipo de dato | Usar un modelo complejo sin saber cuánto aporta | Escalera de referencias de complejidad creciente; regla de un error estándar en B1 (sección 7) |
| **Entrenar** | Resultados reproducibles | Depender de una semilla favorable | Tres semillas; código versionado; configuración y pesos guardados por corrida (sección 7.3) |
| **Interpretar** | Saber qué información usa cada modelo | Un modelo que acierta por razones equivocadas | Permutación, oclusión de bandas y regiones, planos de la cadena y SHAP (sección 8) |
| **Diagnosticar el ajuste** | Detectar sub y sobreajuste | Memorizar el ruido de las etiquetas o no captar la señal | Entrenamiento frente a validación, curvas de aprendizaje y de entrenamiento, residuos (sección 9) |
| **Evaluar** | Estimación honesta del desempeño | Ajustar decisiones al conjunto de prueba | Elección registrada con huella; prueba abierta una vez; comparación pareada con Katachi (sección 10) |

**Respuestas a las preguntas del avance.**

_Se completan con los resultados de la corrida:_

1. ¿Qué algoritmo se puede utilizar como baseline?
2. ¿Se puede determinar la importancia de las características?
3. ¿El modelo está sub o sobreajustando?
4. ¿Cuál es la métrica adecuada?
5. ¿Cuál debería ser el desempeño mínimo?

## 12. Referencias

**Metodología**

- Breiman, L. (2001). Random forests. *Machine Learning*, 45, 5–32.
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3.ª ed.). O'Reilly Media.
- Hastie, T., Tibshirani, R. y Friedman, J. (2009). *The Elements of Statistical Learning* (2.ª ed.). Springer.
- He, K., Zhang, X., Ren, S. y Sun, J. (2016). Deep residual learning for image recognition. *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition*, 770–778.
- Lundberg, S. M. y Lee, S.-I. (2017). A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems*, 30.
- Studer, S., Bui, T. B., Drescher, C., et al. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. *Machine Learning and Knowledge Extraction*, 3, 392–413.

**Astronomía**

- Alfonzo, J. P., Iyer, K. G., Akiyama, M., et al. (2024). Decoding the imprints of past star formation on present-day morphology in galaxies with interpretable CNNs. *The Astrophysical Journal*, 967, 152.
- Blanton, M. R., Kazin, E., Muna, D., Weaver, B. A. y Price-Whelan, A. (2011). Improved background subtraction for the Sloan Digital Sky Survey images. *The Astronomical Journal*, 142, 31.
- Brammer, G. B., van Dokkum, P. G. y Coppi, P. (2008). EAZY: A fast, public photometric redshift code. *The Astrophysical Journal*, 686, 1503.
- Dey, A., Schlegel, D. J., Lang, D., et al. (2019). Overview of the DESI Legacy Imaging Surveys. *The Astronomical Journal*, 157, 168.
- Roediger, J. C. y Courteau, S. (2015). On the uncertainties of stellar mass estimates via colour measurements. *Monthly Notices of the Royal Astronomical Society*, 452, 3209.
- Sánchez, S. F., Barrera-Ballesteros, J. K., Lacerda, E., et al. (2022). SDSS-IV MaNGA: Pipe3D analysis release for 10,000 galaxies. *The Astrophysical Journal Supplement Series*, 262, 36.
- Taylor, E. N., Hopkins, A. M., Baldry, I. K., et al. (2011). Galaxy And Mass Assembly (GAMA): stellar mass estimates. *Monthly Notices of the Royal Astronomical Society*, 418, 1587.